In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("customer_churn_numeric.csv")
print(df)

      customerID  SeniorCitizen  tenure  MonthlyCharges  TotalCharges  \
0           5375            0.0     1.0           29.85          2505   
1           3962            0.0    34.0           56.95          1466   
2           2564            0.0     2.0           53.85           157   
3           5535            0.0    45.0           42.30          1400   
4           6511            0.0     2.0           70.70           925   
...          ...            ...     ...             ...           ...   
7038        4853            0.0    24.0           84.80          1597   
7039        1525            0.0    72.0          103.20          5698   
7040        3367            0.0    11.0           29.60          2994   
7041        5934            0.0     4.0           74.40          2660   
7042        2226            0.0    66.0          105.65          5407   

      gender_Male  Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0               0            1               0         

In [3]:
df.isnull().sum()

customerID                               0
SeniorCitizen                            0
tenure                                   0
MonthlyCharges                           0
TotalCharges                             0
gender_Male                              0
Partner_Yes                              0
Dependents_Yes                           0
PhoneService_Yes                         0
MultipleLines_No phone service           0
MultipleLines_Yes                        0
InternetService_Fiber optic              0
InternetService_No                       0
OnlineSecurity_No internet service       0
OnlineSecurity_Yes                       0
OnlineBackup_No internet service         0
OnlineBackup_Yes                         0
DeviceProtection_No internet service     0
DeviceProtection_Yes                     0
TechSupport_No internet service          0
TechSupport_Yes                          0
StreamingTV_No internet service          0
StreamingTV_Yes                          0
StreamingMo

In [4]:
df.isnull().mean()*100

customerID                               0.0
SeniorCitizen                            0.0
tenure                                   0.0
MonthlyCharges                           0.0
TotalCharges                             0.0
gender_Male                              0.0
Partner_Yes                              0.0
Dependents_Yes                           0.0
PhoneService_Yes                         0.0
MultipleLines_No phone service           0.0
MultipleLines_Yes                        0.0
InternetService_Fiber optic              0.0
InternetService_No                       0.0
OnlineSecurity_No internet service       0.0
OnlineSecurity_Yes                       0.0
OnlineBackup_No internet service         0.0
OnlineBackup_Yes                         0.0
DeviceProtection_No internet service     0.0
DeviceProtection_Yes                     0.0
TechSupport_No internet service          0.0
TechSupport_Yes                          0.0
StreamingTV_No internet service          0.0
StreamingT

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from thefuzz import process
import warnings

warnings.filterwarnings('ignore')

# --- 1. FEATURES LIST ---
FEATURES = [
    'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
    'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
    'MultipleLines_No phone service', 'MultipleLines_Yes',
    'InternetService_Fiber optic', 'InternetService_No',
    'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
    'OnlineBackup_No internet service', 'OnlineBackup_Yes',
    'DeviceProtection_No internet service', 'DeviceProtection_Yes',
    'TechSupport_No internet service', 'TechSupport_Yes',
    'StreamingTV_No internet service', 'StreamingTV_Yes',
    'StreamingMovies_No internet service', 'StreamingMovies_Yes',
    'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
    'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check',
    'PaymentMethod_Mailed check'
]

# --- 2. UPDATED SMART ALIGNMENT FUNCTION ---
def align_client_data_smart(client_df, model_features):
    """
    This function done autonomously the work.
    """
    client_df.columns = [c.lower() for c in client_df.columns]

    # 1. INDUSTRY SYNONYMS (AI Common Sense)
    industry_keys = {
        'MonthlyCharges': ['balance', 'monthly_spend', 'amount', 'charges', 'salary', 'monthlycharges', 'spent'],
        'tenure': ['months', 'period', 'account_age', 'membership', 'days', 'tenure', 'duration'],
        'TotalCharges': ['total_balance', 'total_spent', 'cumulative', 'totalcharges', 'total_revenue'],
        'SeniorCitizen': ['age', 'is_old', 'senior', 'seniorcitizen', 'elderly'],
        'gender_Male': ['sex', 'is_male', 'gender', 'male']
    }

    mapped_df = pd.DataFrame(index=client_df.index)

    # firstly find the features (Bank vs Ecommerce)
    for target, alternatives in industry_keys.items():
        for alt in alternatives:
            if alt in client_df.columns:
                mapped_df[target] = client_df[alt]
                break

    # B.  Fuzzy Matching for rest of 30+ features
    client_cols = list(client_df.columns)
    for feature in model_features:
        if feature not in mapped_df.columns:
            # Lowercasing and find the match
            match, score = process.extractOne(feature.lower(), client_cols)
            if score > 80:
                mapped_df[feature] = client_df[match]
            else:
                mapped_df[feature] = 0 # Default if no match found

    # C. Ensure Correct Sequence & Numeric Conversion
    final_df = mapped_df.reindex(columns=model_features).fillna(0)
    return final_df.apply(pd.to_numeric, errors='coerce').fillna(0)

# --- 3. PREPROCESSING & MODEL TRAINING ---
# 'df' is your training dataset in this case  (e.g. Ecommerce data)
X = align_client_data_smart(df, FEATURES)
y = df['churn_yes'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ANN ARCHITECTURE
model_ann = Sequential([
    Dense(64, activation='relu', input_shape=(len(FEATURES),)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_ann.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=1)

# --- 4. PREDICTION FUNCTION ---
def predict_churn_ann_final(new_client_csv):
    # 1. Load Data
    client_data = pd.read_csv(new_client_csv)

    # 2. Smart Alignment
    ready_data = align_client_data_smart(client_data, FEATURES)

    # 3. Scaling
    ready_data_scaled = scaler.transform(ready_data)

    # 4. Predict
    probabilities = model_ann.predict(ready_data_scaled).flatten()

    # 5. Format Results
    client_data['Risk_Score'] = np.round(probabilities, 2)

    client_data['Status'] = np.where(probabilities > 0.5, '⚠️ High Risk', '✅ Safe')

    return client_data

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# 1.check the training histoey of model
# 2. check the performance on test data
y_pred_prob = model_ann.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\n--- Model Evaluation Report ---")
print(classification_report(y_test, y_pred))

# 3. Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Safe', 'Churn'], yticklabels=['Safe', 'Churn'])
plt.title('ANN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
import pickle

# 1. ANN Model save (H5,pkl format )
model_ann.save('churn_model_ann.h5')
print("✅ ANN Model saved as 'churn_model_ann.h5'")

# 2. Scaler save
with open('scaler_ann.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved as 'scaler_ann.pkl'")

# 3. save the frature list also
with open('features_list.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

In [ ]:
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

# --- 5. FINAL UNIVERSAL PREDICTION FUNCTION (FIXED) ---
def get_final_predictions(new_csv_path):
    # 1. Load Model
    loaded_model = load_model('churn_model_ann.h5')
    print("✅ Model Loaded Successfully")

    # 2. Load Scaler
    with open('scaler_ann.pkl', 'rb') as f:
        loaded_scaler = pickle.load(f)
    print("✅ Scaler Loaded Successfully")

    # 3. Load New Client Data
    client_data = pd.read_csv(new_csv_path)

    client_data.columns = [c.lower() for c in client_data.columns]

    # 4. Smart Alignment (Important: Use FEATURES from training)
    ready_data = align_client_data_smart(client_data, FEATURES)

    # 5. Scaling
    ready_data_scaled = loaded_scaler.transform(ready_data)

    # 6. Predict
    probabilities = loaded_model.predict(ready_data_scaled).flatten()

    print(f"Maximum Probability: {probabilities.max():.4f}")
    print(f"Average Probability: {probabilities.mean():.4f}")

    # 7. DYNAMIC THRESHOLD LOGIC (The Fix)
    if probabilities.max() < 0.5:
        dynamic_threshold = np.percentile(probabilities, 85)
        print(f"⚠️ Low confidence detected. Using Dynamic Threshold: {dynamic_threshold:.4f}")
    else:
        dynamic_threshold = 0.5
        print(f"ℹ️ Standard Threshold (0.5) used.")

    # 8. Formatting Results
    client_data['risk_score'] = np.round(probabilities, 4)
    client_data['status'] = np.where(probabilities >= dynamic_threshold, '⚠️ High Risk', '✅ Safe')

    # Return Results
    cols_to_show = ['customerid', 'risk_score', 'status']
    existing_cols = [c for c in cols_to_show if c in client_data.columns]

    return client_data[existing_cols] if existing_cols else client_data

# --- TEST IT ---
try:
    file_path = 'Bank_customer_churn_prediction.csv'

    final_df = get_final_predictions(file_path)

    print("\n--- Final Predictions for Bank Data ---")
    print(final_df.head(10))

    print("\n--- Summary Counts ---")
    print(final_df['status'].value_counts())

except Exception as e:
    print(f"❌ Error occurred: {e}")

In [ ]:
def get_final_predictions_advanced(new_csv_path):
    # 1. Load Model & Scaler
    loaded_model = load_model('churn_model_ann.h5')
    with open('scaler_ann.pkl', 'rb') as f:
        loaded_scaler = pickle.load(f)

    # 2. Data Load & Align
    client_data = pd.read_csv(new_csv_path)
    client_data.columns = [c.lower() for c in client_data.columns]
    ready_data = align_client_data_smart(client_data, FEATURES)
    ready_data_scaled = loaded_scaler.transform(ready_data)

    # 3. Predict
    probabilities = loaded_model.predict(ready_data_scaled).flatten()

    # 4. ADVANCED SEGMENTATION LOGIC
    low_threshold = np.percentile(probabilities, 70)   # Top 30% users
    high_threshold = np.percentile(probabilities, 90)  # Top 10% users (Most Risky)

    conditions = [
        (probabilities >= high_threshold),
        (probabilities >= low_threshold) & (probabilities < high_threshold),
        (probabilities < low_threshold)
    ]

    # Categories for Client
    choices = ['🔥 Critical Risk', '⚠️ Medium Risk', '✅ Safe']

    # 5. Result Formatting
    client_data['risk_score'] = np.round(probabilities, 4)
    client_data['status'] = np.select(conditions, choices, default='✅ Safe')

    # Return only key columns
    cols_to_show = ['customerid', 'risk_score', 'status']
    existing_cols = [c for c in cols_to_show if c in client_data.columns]
    return client_data[existing_cols] if existing_cols else client_data

# --- TEST IT ---
try:
    final_df = get_final_predictions_advanced('Bank_customer_churn_prediction.csv')
    print("\n--- Advanced Risk Segmentation ---")
    print(final_df.head(15))
    print("\n--- Summary Breakdown ---")
    print(final_df['status'].value_counts())
except Exception as e:
    print(f"❌ Error: {e}")